# Wykład 4: NumPy essentials — analiza obrazów mikroskopowych



## 4.1 Od danych tabelarycznych do struktur wielowymiarowych

W poprzednich wykładach analizowaliśmy dane w postaci **tabel** — wiersze odpowiadały pacjentom, kolumny odpowiadały zmiennym klinicznym. Biblioteka pandas doskonale radzi sobie z tego typu danymi. Ale w biologii i medycynie spotykamy również dane o zupełnie innej strukturze:

* **obrazy mikroskopowe** — macierze pikseli, często wielowarstwowe (RGB, Z-stack, time-lapse),
* **serie czasowe pomiarów** — np. aktywność neuronów w czasie, zmiany temperatury,
* **dane genomiczne** — macierze ekspresji genów (tysiące genów × setki próbek),
* **dane obrazowe medyczne** — MRI, CT, obrazy histopatologiczne.

Do tego typu danych potrzebujemy biblioteki **NumPy** (*Numerical Python*), która jest fundamentem obliczeń numerycznych w Pythonie. NumPy wprowadza strukturę `ndarray` (*n-dimensional array*) — wielowymiarową tablicę liczb o ustalonej wielkości i typie.

Dzisiaj nauczymy się podstaw NumPy analizując rzeczywiste dane mikroskopowe: **time-lapse podziału komórkowego (mitozy)** zarejestrowany w mikroskopie fluorescencyjnym.



## 4.2 Problem badawczy: Mitoza w mikroskopie

**Mitoza** to etap cyklu komórkowego, w którym skondensowane chromosomy są rozdzielane do dwóch jąder potomnych. To kluczowy proces dla wzrostu i odnowy tkanek — każda komórka naszego ciała (poza gametami) powstała właśnie w wyniku mitozy.

W tym przykładzie wykorzystujemy plik `mitosis.tif` — jeden z **przykładowych datasetów ImageJ/Fiji**, powszechnie używany jako materiał edukacyjny do pracy z danymi wielowymiarowymi w mikroskopii.  
Źródło: https://imagej.net/ij/images/ (plik: [mitosis.tif](https://imagej.net/ij/images/mitosis.tif))

Dane mają postać **obrazu wielowymiarowego (hyperstack)** z osiami **T (czas), Z (przekroje), C (kanały) oraz XY**:

- **T = 51 time-pointów** — kolejne klatki nagrania time-lapse. Dzięki nim możemy śledzić dynamikę podziału: kondensację chromosomów, formowanie wrzeciona, rozdzielanie chromatyd i dekondensację. (Uwaga: metadane pliku zawierają parametr `finterval`, ale bez jednoznacznie opisanej jednostki — dlatego podajemy liczbę klatek, nie czas trwania w minutach.)
- **Z = 5 przekrojów optycznych** — komórka jest obiektem trójwymiarowym, więc mikroskop rejestruje obraz na kilku głębokościach ogniskowania (tzw. Z-stack). Pozwala to uchwycić struktury leżące w różnych płaszczyznach, np. chromosomy u góry i u dołu wrzeciona podziałowego.
- **C = 2 kanały fluorescencyjne** — każdy kanał odpowiada innemu fluoroforowi, czyli barwnikowi wiążącemu się z określoną strukturą komórkową. W tym preparacie jeden kanał uwidacznia chromosomy/centrosomy, drugi — mikrotubule wrzeciona podziałowego.
- Pojedyncza klatka ma rozmiar **Y = 196 × X = 171 pikseli**; typ piksela to **16-bit (uint16)**.

W efekcie jest to **5-wymiarowy dataset**: **czas (T) × głębokość (Z) × kanał (C) × wysokość (Y) × szerokość (X)**.

**Pytania badawcze:**

* Kiedy chromosomy się rozdzielają?
* Jak wygląda wrzeciono podziałowe w 3D?
* Które struktury są najbardziej dynamiczne?
* Jak „uśrednić" wiele klatek, żeby zmniejszyć szum?



## 4.3 Pierwsze kroki z NumPy



### 4.3.1 Import i wczytanie danych



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tf


NumPy importujemy tradycyjnie jako `np`. Biblioteka `tifffile` służy do wczytywania plików TIFF — popularnego formatu w mikroskopii.

Wczytajmy dane:



In [ ]:
with tf.TiffFile("data/mitosis.tif") as tif:
    data = tif.asarray()
    axes = tif.series[0].axes

print(f"Axes: {axes}")
print(f"Shape: {data.shape}")

**Interpretacja:**

* `axes = 'TZCYX'` — oznacza kolejność wymiarów: **T**ime, **Z**-depth, **C**hannel, **Y**, **X**
* `shape = (51, 5, 2, 196, 171)` oznacza:
  * 51 momentów czasowych (time-points)
  * 5 warstw Z (przekroje optyczne)
  * 2 kanały fluorescencyjne
  * 196 pikseli wysokości
  * 171 pikseli szerokości

To jest **5-wymiarowy tensor** — struktura, której pandas DataFrame nie potrafi obsłużyć. NumPy radzi sobie z tym bez problemu.



### 4.3.2 Podstawowe właściwości array

Każdy array NumPy ma kilka kluczowych atrybutów:



In [ ]:
print(f"Wymiary (shape): {data.shape}")
print(f"Liczba wymiarów (ndim): {data.ndim}")
print(f"Typ danych (dtype): {data.dtype}")
print(f"Liczba elementów (size): {data.size:,}")
print(f"Rozmiar w pamięci: {data.nbytes / (1024**2):.1f} MB")

**Uwagi:**

* `dtype = uint16` — liczby całkowite bez znaku 16-bitowe (zakres 0–65535), typowe dla kamer mikroskopowych.
* `size` — całkowita liczba elementów (51 × 5 × 2 × 196 × 171).
* Dataset jest kompaktowy (32.6 MB) — mieści się w pamięci.



## 4.4 Praca z pojedynczym obrazkiem (2D array)

Zacznijmy od czegoś prostego: wyciągnijmy **jeden obrazek** z tego 5D datasetu. Wybieramy środkowy moment czasowy (t=25, aktywna mitoza) i środkową warstwę optyczną (z=2):



In [ ]:
frame_c0 = data[25, 2, 0, :, :]
frame_c1 = data[25, 2, 1, :, :]

print(f"Shape pojedynczego obrazka: {frame_c0.shape}")

Mamy teraz dwa **2D array** — dokładnie to, czym jest cyfrowy obraz (macierz pikseli). Ale który kanał co pokazuje? Metadane pliku mówią nam, że są dwa kanały fluorescencyjne — **nie mówią** jednak, jaki barwnik odpowiada jakiemu kanałowi. To informacja, która powinna towarzyszyć eksperymentowi (protokół barwienia, notatki badacza). Zróbmy to, co robi mikroskopista, gdy dostaje nowy dataset — **po prostu popatrzmy**:



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(frame_c0, cmap="gray")
axes[0].set_title("Kanał 0")
axes[0].axis("off")

axes[1].imshow(frame_c1, cmap="gray")
axes[1].set_title("Kanał 1")
axes[1].axis("off")

plt.tight_layout()
plt.show()


**Co widzimy?**

- **Kanał 0** — jasne, kompaktowe punkty skupione w centrum. To charakterystyczny obraz **skondensowanych chromosomów** (lub centrosomów) w trakcie podziału.
- **Kanał 1** — wydłużona, wrzecionowata struktura przypominająca piłkę do rugby, złożona z grubych włókien. To typowy obraz **wrzeciona podziałowego** — aparatu zbudowanego z mikrotubul, który rozciąga chromosomy do przeciwnych biegunów komórki.

To przypisanie jest **hipotezą opartą na wiedzy biologicznej**, nie na metadanych pliku. W prawdziwej pracy laboratoryjnej taką interpretację potwierdzilibyśmy sięgając do protokołu eksperymentu (jakie barwniki zostały użyte, np. DAPI na DNA, GFP-tubulina na mikrotubule).

Od tego momentu będziemy używać roboczych oznaczeń: kanał 0 = „chromosomy", kanał 1 = „wrzeciono", pamiętając o tym zastrzeżeniu.

Dalszą analizę przeprowadzimy na kanale chromosomowym:



In [ ]:
frame = frame_c0  # kanał 0 — chromosomy (nasza hipoteza)


### 4.4.1 Statystyki podstawowe



In [ ]:
print(f"Wartość minimalna: {frame.min()}")
print(f"Wartość maksymalna: {frame.max()}")
print(f"Średnia: {frame.mean():.2f}")
print(f"Odchylenie standardowe: {frame.std():.2f}")
print(f"Mediana: {np.median(frame):.2f}")

**Interpretacja biologiczna:**

- Wartości to intensywność fluorescencji — wyższe wartości oznaczają więcej barwnika, czyli więcej skondensowanej chromatyny.
- Średnia (2311) jest wyraźnie wyższa od mediany (2059), a maksimum (30549) to ponad 13-krotność średniej — rozkład jest **prawoskośny**. To ma sens: większość pikseli to ciemne tło (niskie wartości), a nieliczne piksele chromosomów tworzą długi jasny ogon.

### 4.4.2 Wizualizacja



In [ ]:
plt.figure(figsize=(8, 7))
plt.imshow(frame, cmap="gray")
plt.colorbar(label="Intensywność fluorescencji")
plt.title("Chromosomy w momencie t=25, warstwa Z=2")
plt.xlabel("X (piksele)")
plt.ylabel("Y (piksele)")
plt.show()


Na obrazku widać dwa jasne skupiska — to są dwie grupy chromosomów, które właśnie się rozdzielają podczas mitozy.



### 4.4.3 Boolean indexing — segmentacja chromosomów

Chcemy oddzielić piksele chromosomów od tła. Wiemy, że chromosomy świecą jaśniej — wystarczy wybrać piksele powyżej pewnego progu:



In [ ]:
threshold = 3000
bright_mask = frame > threshold

print(f"Typ maski: {bright_mask.dtype}")
print(f"Shape maski: {bright_mask.shape}")
print(f"Liczba jasnych pikseli: {bright_mask.sum()} / {frame.size}")
print(f"Procent: {100 * bright_mask.sum() / frame.size:.1f}%")

**Co się stało?**

* `frame > threshold` zwraca array boolowski — `True` tam, gdzie warunek spełniony, `False` gdzie nie.
* `.sum()` na array boolowskim liczy `True` jako 1, `False` jako 0 → daje liczbę pikseli jasnych.

Możemy teraz **wyekstrahować wartości** tylko jasnych pikseli:



In [ ]:
bright_values = frame[bright_mask]
# Krótsza wersja: bright_values = frame[frame > threshold]

print(f"Shape wyekstrahowanych wartości: {bright_values.shape}")
print(f"Średnia jasnych pikseli: {bright_values.mean():.2f}")
print(f"Min jasnych: {bright_values.min()}, Max jasnych: {bright_values.max()}")

Piksele powyżej progu mają średnią 6433 — prawie 3× więcej niż średnia całości (2311). To właśnie tam są chromosomy.

**Wizualizacja maski:**



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(frame, cmap="gray")
axes[0].set_title("Oryginalny obraz")
axes[0].axis("off")

axes[1].imshow(bright_mask, cmap="gray")
axes[1].set_title(f"Maska: piksele > {threshold}")
axes[1].axis("off")

plt.tight_layout()
plt.show()


Maska pokazuje tylko regiony z chromosomami — reszta to tło.



### 4.4.4 Slicing — Region of Interest (ROI)

Na masce widać, gdzie leżą chromosomy. Możemy wyciąć ten region bezpośrednio z obrazu, używając składni `[start:stop]` (podobnie jak w listach Pythona, ale w dwóch wymiarach):



In [ ]:
# Region centralny — obejmuje oba skupiska chromosomów
roi = frame[80:116, 70:101]  # [y_start:y_stop, x_start:x_stop]
print(f"ROI shape: {roi.shape}")
print(f"ROI średnia: {roi.mean():.2f}")
print(f"Cały obraz średnia: {frame.mean():.2f}")

ROI ma prawie 3× wyższą średnią intensywność niż cały obraz — potwierdzenie, że trafiliśmy w region chromosomów. W dalszej części wykładu zobaczymy, jak śledzić taki ROI w czasie.



## 4.5 Operacje po osiach — parameter `axis`

Do tej pory pracowaliśmy z pojedynczym obrazem 2D. Teraz wracamy do pełnych danych czasowych — chcemy odpowiedzieć na pytania biologiczne, które wymagają analiz **wzdłuż wybranych wymiarów**.

Kluczowa koncepcja: **`axis` określa, który wymiar „znika" po agregacji.**

Wyciągnijmy serię czasową — 51 obrazów chromosomów z jednej warstwy Z:



In [ ]:
time_series = data[:, 2, 0, :, :]  # (51, 196, 171) = (czas, Y, X)
print(f"Time-series shape: {time_series.shape}")

Mamy teraz 3D array: 51 klatek, każda 196×171 pikseli. Zobaczmy, co możemy z tego wyciągnąć.



### 4.5.1 Średni obraz — `mean(axis=0)`

Chcemy zobaczyć, jak wygląda „przeciętna" klatka tego time-lapse'u. Uśredniamy po osi czasu (axis=0) — czas „znika", zostaje obraz (Y, X):



In [ ]:
mean_image = time_series.mean(axis=0)
print(f"Średnia po czasie: {mean_image.shape}")  # (196, 171)

plt.figure(figsize=(8, 7))
plt.imshow(mean_image, cmap="gray")
plt.colorbar(label="Średnia intensywność")
plt.title("Średni obraz przez 51 time-pointów")
plt.show()

Struktury, które nie poruszają się w trakcie eksperymentu (np. centrosomy, jeśli są nieruchome), będą widoczne ostro. Struktury, które się przemieszczają (chromosomy wędrujące do biegunów), będą rozmyte — ich jasność „rozmazuje się" po wielu pozycjach.



### 4.5.2 Mapa zmienności — `std(axis=0)`

Średnia mówi, gdzie *coś jest*. Odchylenie standardowe mówi, gdzie *coś się dzieje*:



In [ ]:
std_map = time_series.std(axis=0)

plt.figure(figsize=(8, 7))
plt.imshow(std_map, cmap="hot")
plt.colorbar(label="Odchylenie standardowe")
plt.title("Które piksele się zmieniają w czasie?")
plt.show()


Wysokie wartości std = ten piksel zmienia jasność w kolejnych klatkach (ruch chromosomów, pojawianie się i zanikanie struktur). Niskie std = tło, nic się tam nie dzieje. To prosta, ale potężna metoda detekcji ruchu w time-lapse'ach.



### 4.5.3 Profil temporalny — `mean(axis=(1,2))`

Odwrotne pytanie: jak zmienia się **globalna jasność** klatka po klatce? Tym razem uśredniamy po przestrzeni (osie Y i X) — przestrzeń „znika", zostaje czas:



In [ ]:
intensity_over_time = time_series.mean(axis=(1, 2))
print(f"Średnia po przestrzeni: {intensity_over_time.shape}")  # (51,)

plt.figure(figsize=(10, 4))
plt.plot(intensity_over_time, linewidth=2)
plt.xlabel("Time point")
plt.ylabel("Średnia intensywność")
plt.title("Jak zmienia się jasność komórki w czasie?")
plt.grid(True, alpha=0.3)
plt.show()

Jeśli widzimy systematyczny spadek intensywności w czasie, to prawdopodobnie **photobleaching** — fluorofory blakną (ulegają degradacji) pod wpływem światła wzbudzającego. To częsty artefakt w mikroskopii fluorescencyjnej.


### 4.5.4 Max projection — technika mikroskopowa

Wróćmy do wymiaru Z. Mamy 5 przekrojów optycznych, ale struktura 3D komórki jest trudna do pokazania na jednym obrazie. Standardowe rozwiązanie w mikroskopii to **max projection** — dla każdego piksela (x, y) bierzemy jego maksymalną wartość spośród wszystkich warstw Z:



In [ ]:
# 5 warstw Z dla momentu t=25, kanał 1 (wrzeciono)
z_stack = data[25, :, 1, :, :]  # Shape: (5, 196, 171)

# Max projection przez Z
max_proj = z_stack.max(axis=0)  # Shape: (196, 171)

# Wizualizacja: wszystkie warstwy + max projection
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

for z in range(5):
    axes[z].imshow(z_stack[z], cmap="gray")
    axes[z].set_title(f"Z={z}")
    axes[z].axis("off")

axes[5].imshow(max_proj, cmap="gray")
axes[5].set_title("Max projection")
axes[5].axis("off")

plt.tight_layout()
plt.show()

**Dlaczego to działa?**

* Struktury fluorescencyjne świecą najbardziej, gdy są w ostrości (w „swoim" przekroju Z).
* `max(axis=0)` dla każdego piksela wybiera najjaśniejszy przekrój.
* Efekt: widzimy **całą strukturę 3D spłaszczoną do 2D** — każdy element w swoim najlepszym momencie.

To standard w mikroskopii — większość publikacji pokazuje max projections zamiast pojedynczych przekrojów.



## 4.6 Plotly — interaktywna eksploracja

Do tej pory używaliśmy matplotlib do statycznych wykresów. Teraz poznamy **Plotly** — bibliotekę do interaktywnych wizualizacji, która zmienia sposób eksploracji danych obrazowych.



In [ ]:
import plotly.express as px


### 4.6.1 Pojedynczy obraz



In [ ]:
frame = data[25, 2, 0, :, :]  # Chromosomy, t=25, Z=2

fig = px.imshow(
    frame,
    color_continuous_scale="gray",
    labels={"x": "X", "y": "Y"},
    title="Chromosomy (interaktywne!)",
)
fig.show()

**Co możemy zrobić w Plotly?**

* Zoom in/out (scroll lub box select)
* Pan (przeciąganie)
* Hover — pokazuje współrzędne i wartość piksela
* Save image (ikonka aparatu)

Hover jest szczególnie przydatny — zamiast zgadywać „czy ten piksel jest jasny", widzimy dokładną wartość. Możemy eksplorować obraz interaktywnie, szukając progów dla segmentacji.



### 4.6.2 Animacja przez czas — time-lapse!

To jest **najciekawsza** część. Plotly może animować przez dowolny wymiar:



In [ ]:
time_series = data[:, 2, 0, :, :]  # (51, 196, 171) — chromosomy w czasie

fig = px.imshow(
    time_series,
    animation_frame=0,  # Animuj po wymiarze 0 (czas)
    color_continuous_scale="gray",
    labels={"animation_frame": "Time point"},
    title="Mitoza w czasie — chromosomy się rozdzielają!",
)

# Ustalamy zakres kolorów (żeby nie migało przy zmianach jasności)
fig.update_layout(
    coloraxis=dict(cmin=time_series.min(), cmax=np.percentile(time_series, 99))
)

fig.show()

Uruchamiamy animację i widzimy jak chromosomy się rozdzielają — to jest moment, w którym dane przestają być abstrakcją, a stają się biologią.

**Funkcjonalności:**

* Play/Pause
* Slider — przeskakujemy do dowolnego momentu
* Frame rate control



### 4.6.3 Animacja przez Z-stack

Możemy też animować przez głębokość — to jak przewijanie mikroskopu przez fokus:



In [ ]:
z_stack = data[25, :, 1, :, :]  # (5, 196, 171) — wrzeciono, różne Z

fig = px.imshow(
    z_stack,
    animation_frame=0,
    color_continuous_scale="gray",
    labels={"animation_frame": "Z slice"},
    title="Wrzeciono podziałowe — przewijanie przez głębokość",
)
fig.show()


Widzimy jak fokus zmienia się — najpierw rozmyte, potem ostre, znowu rozmyte. To dokładnie to, co widzi mikroskopista kręcąc pokrętłem ostrości.



### 4.6.4 Porównanie kanałów

Na koniec zobaczmy oba kanały obok siebie w wersji interaktywnej:



In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

t, z = 25, 2
ch0 = data[t, z, 0, :, :]  # Chromosomy
ch1 = data[t, z, 1, :, :]  # Wrzeciono

fig = make_subplots(
    rows=1, cols=2, subplot_titles=("Kanał 0: Chromosomy", "Kanał 1: Wrzeciono")
)

fig.add_trace(go.Heatmap(z=ch0, colorscale="Greys"), row=1, col=1)
fig.add_trace(go.Heatmap(z=ch1, colorscale="Greys"), row=1, col=2)

fig.update_layout(title=f"Oba kanały (t={t}, Z={z})", height=400)

fig.show()

## 4.7 Podsumowanie



### Co umiemy po wykładzie 4

**NumPy — podstawy:**

✅ Shape, dtype, size — podstawowe właściwości arrays  
✅ Indeksowanie 5D tensora: `data[t, z, c, :, :]`  
✅ Boolean indexing: `arr[arr > threshold]` — segmentacja chromosomów  
✅ Slicing: `arr[y1:y2, x1:x2]` — wycinanie ROI

**Operacje po osiach:**  

✅ `mean(axis=0)` — średni obraz przez czas  
✅ `std(axis=0)` — mapa zmienności (detekcja ruchu)  
✅ `mean(axis=(1,2))` — profil temporalny (photobleaching)  
✅ `max(axis=0)` — max projection (technika mikroskopowa)

**Wizualizacja:**  

✅ matplotlib — statyczne obrazy i wykresy  
✅ Plotly — interaktywna eksploracja, animacje time-lapse i Z-stack



### Pytania badawcze na następny wykład

Dzisiaj opisywaliśmy to, co widać na obrazach. Następnym razem zaczniemy *mierzyć* i *porównywać*:

* Jak znormalizować 51 klatek, żeby skorygować photobleaching?
* Jak korelują oba kanały — czy tam gdzie są chromosomy, jest też wrzeciono?
* Kiedy *dokładnie* następuje rozdzielenie chromatyd — i jak to zmierzyć automatycznie?



### Źródła i dalsza lektura

**Oficjalna dokumentacja:**
* NumPy docs: https://numpy.org/doc/
* Plotly docs: https://plotly.com/python/

**Tutoriale:**
* NumPy quickstart: https://numpy.org/doc/stable/user/quickstart.html
* Plotly fundamentals: https://plotly.com/python/plotly-fundamentals/

**Książki:**
* Wes McKinney, "Python for Data Analysis" — rozdział o NumPy
* Jake VanderPlas, "Python Data Science Handbook" — NumPy i matplotlib



### Zadania do samodzielnej pracy

1. **Histogram intensywności:**
   Na wykładzie widzieliśmy, że rozkład pikseli jest prawoskośny — ale nie widzieliśmy tego na własne oczy. Zrób histogram intensywności dla `frame` (kanał 0, t=25, Z=2) używając `plt.hist()`. Spróbuj różnych wartości `bins` (50, 100, 256). Czy widzisz dwa „piki" (tło vs chromosomy)?

2. **Threshold optimization:**
   Znajdź optymalny threshold dla segmentacji chromosomów. Sprawdź różne wartości (2000, 2500, 3000, ...) i zobacz, która daje najlepsze wyniki (np. najwięcej pikseli w regionach chromosomów, najmniej w tle).

3. **ROI tracking:**
   Wybierz region wokół jednej grupy chromosomów (np. 50×50 pikseli). Oblicz średnią intensywność w tym ROI dla każdego time-pointu. Wykres: jak zmienia się jasność tego regionu w czasie?

4. **Porównanie kanałów — statystyki:**
   Dla momentu t=25, Z=2: oblicz podstawowe statystyki (mean, std, min, max) osobno dla kanału 0 i kanału 1. Który kanał ma większy rozrzut wartości? Co to mówi o strukturach w tych kanałach?

5. **Max projection obu kanałów:**
   Zrób max projection po Z osobno dla kanału 0 i kanału 1 (w momencie t=25). Wyświetl je obok siebie. Który kanał zyskuje więcej na max projection — chromosomy czy wrzeciono? Dlaczego?